# Revisión General de todo el Sistema

## 1 Definir el Surtido Correcto

- SURTIDO: Surtido Activo, en condiciones de ser comprado en cada sucursal , M_HABILITADO_SUCU, M_LISTO_PARA_VENTA_ACT, M_BAJA = 'N', M_A_DAR_DE_BAJA = 'N'
- CONSIDERA: Artículo, Artículo Sucursal, Sucursal Activo con sus márcas correspondientes. 
- VOLUMEN 01/07/2025: Sucursales 185-> 175, Artículos 10.960-> 8.718, Artículo Sucursal 1.010.827-> 1.109.153, Proveedores: 702-> 595
- ARCHIVO: [diarco_data].src.base_forecast_articulos
- GENERADO POR: 1)[data-sync].[dbo].[SP_BASE_PRODUCTOS_SUCURSAL]  --> 2) Importado por: scripts/push/flujo_push_datos_forecast.py -->obtener_base_productos_vigentes.py
- FRECUENCIA: Todos los Dïas Hábiles a las 7:00 hs

## 2 Traer Stock y Datos Relevantes

- CONSIDERA: Stock, OC Pendientes, Vende x Peso, Factor Venta, Dias_Stock, y Datos SGM Repo!! 
- VOLUMEN 01/07/2025: Sucursales 175-> 119, Artículos 10.960->8.718 -> 10.317, Artículo Sucursal 1.010.827-> 1.109.153  ->603.073, Proveedores: 702-> 595 ->681
- ARCHIVO: [diarco_data].src.base_stock_sucursal
- GENERADO POR: 1)[data-sync].[dbo].[SP_BASE_STOCK]  --> 2) Importado por: scripts/push/flujo_push_datos_forecast.py -->obtener_base_stock.py
- FRECUENCIA: Todos los Dïas Hábiles a las 7:00 hs

In [ ]:
""" SELECT c_sucu_empr, c_articulo, c_proveedor_primario, abastecimiento, cod_cd, habilitado, fecha_registro, fecha_baja, unid_transferencia, q_unid_transferencia, 
pedido_min, frente_lineal, capacid_gondola, stock_minimo, cod_comprador, promocion, active_for_purchase, active_for_sale, active_on_mix, delivered_id, product_base_id, 
own_production, q_factor_compra, full_capacity_pallet, number_of_layers, number_of_boxes_per_layer, fuente_origen, fecha_extraccion, estado_sincronizacion
	FROM src.base_productos_vigentes
	--WHERE c_sucu_empr > 300	
	
ORDER BY 1, 2, 3

limit 1000;


-- SUCURSALES 185 --> 175
SELECT DISTINCT c_sucu_empr
	FROM src.base_productos_vigentes
	ORDER BY 1
-- ARTICULOS 10.960 --> 8718
SELECT DISTINCT c_articulo
	FROM src.base_productos_vigentes
	ORDER BY 1

-- ARTICULO SUCURSAL 1.010.827 --> 1.109.153
SELECT  c_articulo, c_sucu_empr
	FROM src.base_productos_vigentes
	--ORDER BY 1, 2

-- PROVEEDORES 702 -> 595
SELECT  DISTINCT c_proveedor_primario
	FROM src.base_productos_vigentes
	ORDER BY 
"""

### BASE stock
""" 
SELECT codigo_articulo, codigo_sucursal, codigo_proveedor, precio_venta, precio_costo, factor_venta, m_vende_por_peso, venta_unidades_1q, venta_unidades_2q, 
venta_mes_unidades, venta_mes_valorizada, dias_stock, fecha_stock, stock, transfer_pendiente, pedido_pendiente, promocion, lote, validez_lote, stock_reserva, 
validez_promocion, q_dias_stock, q_dias_sobre_stock, i_lista_calculado, pedido_sgm, fuente_origen, fecha_extraccion, estado_sincronizacion
	FROM src.base_stock_sucursal
	
	--WHERE c_sucu_empr > 300	
	
ORDER BY 1, 2
limit 1000;

-- SUCURSALES 185 --> 175  --> 119
SELECT DISTINCT codigo_sucursal
	FROM src.base_stock_sucursal
	ORDER BY 1
-- ARTICULOS 10.960 --> 8718
SELECT DISTINCT codigo_articulo
	FROM src.base_stock_sucursal
	ORDER BY 1

-- ARTICULO SUCURSAL 1.010.827 --> 1.109.153
SELECT  codigo_articulo, codigo_sucursal
	FROM src.base_stock_sucursal
	--ORDER BY 1, 2

-- PROVEEDORES 702 -> 595
SELECT  DISTINCT codigo_proveedor
	FROM src.base_stock_sucursal
	ORDER BY 1
"""

## 3) Seleccionar el Algoritmo y Obtener Datos Básicos

- GENERAR DATOS BASICOS por ID PROVEEDOR: desde funciones_forecast.py / generar_datos(id_proveedor, etiqueta, ventana)
    - OBTENER Artículos activos + stock CONCATENAR y  GRABAR csv intermedio -> _articulos.csv, _stock_sucursal.csv,
- OBTENER VENTAS HISTÓRICAS --> qry -> src.t702_est_vtas_por_articulo (DIARCO) + src.t702_est_vtas_por_articulo_dbarrio (BARRIO)
    - GRABAR csv intermedio -> _Ventas.csv
- INNER JOIN de ARTICULOS y VENTAS --> guarda 2 csv intermedios,  archivo_datos.csv (FULL DATOS)  + _Ventas.csv  (Solo tiene las VENTAS filtradas de los activos)
    - Compastar solo VENTAS en data

## 4) Aplicar Algoritmos s/ FULL DATOS
- GROUP BY Articulo, Sucursal
    Filtrar Lote de Ventas y aplicar diferentes Algoritmos
    Estimar la Demanda y Grabar FORECAST + AVERAGE + VENTANA + Datos Relevantes del Algoritmo aplicado
- GENERAR _ALGO_0x_Solicitudes_Compra.csv 

## 5) Exteneder Datos + Graficar
- DATOS de Stock y Logísticos + ids de Connexa
- JSON: Series de Datos para Gráficos s/cada registro
- JOIN: Siempre LEFT --> Partiendo de la Demanda Calculada y Solo de los que tuvieron VENTAS y están Activos.

## 6) Publicar RESULTADOS en Connexa

